In [ ]:
import pandas as pd
import sys
sys.path.append('..')  
df = pd.read_excel('../data/propostas_concatenadas.xlsx')

In [ ]:
df.columns

In [ ]:
# Create df_clientes with specific columns: Nome do Cliente, CNPJ, Proposta Comercial, Estado, Cidade
df_clientes = df[['Nome do Cliente', 'CNPJ', 'Proposta Comercial', 'Estado', 'Cidade']].copy()
df_clientes.head(10)

In [ ]:
df_clientes['CNPJ'].value_counts()

In [ ]:
# CNPJ duplicado
duplicate_cnpjs = df['CNPJ'].value_counts()
duplicate_cnpjs = duplicate_cnpjs[duplicate_cnpjs > 1]

print(f"{len(duplicate_cnpjs)} CNPJs duplicados:")
print(duplicate_cnpjs)

print("\n" + "="*50)
print("Colunas com cnpj duplicado:")
print("="*50)

# Mostra coluna com cnpj duplicado
for cnpj in duplicate_cnpjs.index:
    print(f"\nCNPJ: {cnpj} (Aparece {duplicate_cnpjs[cnpj]} vezes)")
    duplicate_rows = df[df['CNPJ'] == cnpj]
    display(duplicate_rows)

In [ ]:
# Importar db 
from dicts.db_extractor import load_database

# Load db 
df_clientes_db = load_database('../data/dump_clientes.sql')

# Mostrar db 
df_clientes_db.head()

In [ ]:
# Limpar e padronizar CNPJs
def clean_cnpj(cnpj_series):
    #Remove espaços, caracteres especiais e padroniza CNPJs
    return cnpj_series.dropna().astype(str).str.strip().str.replace(r'[^\d]', '', regex=True)

# Criar colunas de CNPJ limpo
df_clientes['CNPJ_limpo'] = clean_cnpj(df_clientes['CNPJ'])
df_clientes_db['CNPJ_limpo'] = clean_cnpj(df_clientes_db['CPF_CNPJ'])

# Encontrar CNPJs que existem em ambos
excel_cnpjs_clean = set(df_clientes['CNPJ_limpo'])
db_cnpjs_clean = set(df_clientes_db['CNPJ_limpo'])
matching_cnpjs = excel_cnpjs_clean.intersection(db_cnpjs_clean)

print(f"CNPJs que existem em AMBOS os datasets: {len(matching_cnpjs)}")

In [ ]:
# Mostrar comparação dos CNPJs matching
if matching_cnpjs:
    comparison_data = []
    
    # Filtrar CNPJs válidos (apenas strings com pelo menos 11 dígitos)
    valid_cnpjs = [cnpj for cnpj in matching_cnpjs if isinstance(cnpj, str) and cnpj.isdigit() and len(cnpj) >= 11]
    
    for cnpj_clean in sorted(valid_cnpjs):
        # Buscar dados do Excel
        excel_matches = df_clientes[df_clientes['CNPJ_limpo'] == cnpj_clean]
        
        # Buscar dados da Base de Dados
        db_matches = df_clientes_db[df_clientes_db['CNPJ_limpo'] == cnpj_clean]
        
        if len(excel_matches) > 0 and len(db_matches) > 0:
            excel_data = excel_matches.iloc[0]
            db_data = db_matches.iloc[0]
            
            comparison_data.append({
                'CNPJ': cnpj_clean,
                'Nome_Excel': excel_data['Nome do Cliente'],
                'Nome_BD': db_data['Nome'],
            })
    
    # Mostrar tabela de comparação
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        display(comparison_df)
        print(f"\nTotal de empresas encontradas em ambos os datasets: {len(comparison_data)}")
    else:
        print("Nenhum CNPJ válido encontrado para comparação")
        
else:
    print("Nenhum CNPJ encontrado em ambos os datasets")

In [ ]:
print("Quantidade de clientes no xlsx:", len(df_clientes))

In [ ]:
1137 - 201